# Dataset preparation for AMR prediction

In order to reduce the risk of model over-parameterization, the training dataset was expanded by incorporating additional samples from multiple centers.

Previously, only **DRIAMS-A** spectra were used. In this analysis we include:

- **DRIAMS-A, B and C**
- **Marisma clinical dataset**

Both datasets were preprocessed and stored as serialized **pickle files** containing:

- MALDI-TOF spectra
- species labels
- antimicrobial resistance (AMR) labels
- antibiotic metadata

# Dataset preparation for AMR prediction

In order to reduce the risk of model over-parameterization, the training dataset was expanded by incorporating additional samples from multiple centers.

Previously, only **DRIAMS-A** spectra were used. In this analysis we include:

- **DRIAMS-A, B and C**
- **Marisma clinical dataset**

Both datasets were preprocessed and stored as serialized **pickle files** containing:

- MALDI-TOF spectra
- species labels
- antimicrobial resistance (AMR) labels
- antibiotic metadata

## Imports and configuration

In [ ]:
import pickle
import numpy as np
import pandas as pd
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, accuracy_score, hamming_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

driams_path = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/DRIAMS_A_AMR_whole_pipeline_23_DRIAMS.pkl"
marisma_path = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/MARISMa_study_MARISMA_whole_pipeline.pkl"

with open(driams_path, "rb") as f:
    driams_payload = pickle.load(f)

with open(marisma_path, "rb") as f:
    marisma_payload = pickle.load(f)

print("Datasets loaded.")


def inspect_dataset(payload, name):

    X = payload["data"]
    y_species = payload["label"]
    amr = payload["amr"]
    antibiotics = payload["antibiotics"]

    print(f"\n{name}")
    print("-" * 40)
    print("Spectra shape:", X.shape)
    print("AMR matrix shape:", amr.shape)
    print("Number of antibiotics:", len(antibiotics))
    print("Unique species:", np.unique(y_species))


inspect_dataset(driams_payload, "DRIAMS ABC")
inspect_dataset(marisma_payload, "MARISMA")

Using device: cpu
Datasets loaded.

DRIAMS ABC
----------------------------------------
Spectra shape: (18168, 6000)
AMR matrix shape: (18168, 9)
Number of antibiotics: 9
Unique species: ['Escherichia_Coli' 'Klebsiella_Pneumoniae' 'Pseudomonas_Aeruginosa'
 'Staphylococcus_Aureus']

MARISMA
----------------------------------------
Spectra shape: (21360, 6000)
AMR matrix shape: (21360, 8)
Number of antibiotics: 8
Unique species: ['Escherichia_Coli' 'Klebsiella_Pneumoniae' 'Pseudomonas_Aeruginosa'
 'Staphylococcus_Aureus']


## Species specific antibiotic definition

In [ ]:
species_antibiotics = {
    "Staphylococcus_Aureus": [
        "Oxacillin", "Clindamycin", "Fusidic acid"
    ],
    "Escherichia_Coli": [
        "Ciprofloxacin", "Ceftriaxone",
        #"Piperacillin-Tazobactam",
		 "Cefepime"
    ],
    "Klebsiella_Pneumoniae": [
        "Ciprofloxacin", #"Ceftriaxone",
        "Imipenem", "Meropenem"
    ],
    "Pseudomonas_Aeruginosa": [
        "Ciprofloxacin", "Imipenem", "Meropenem"
    ]
}

## Generate species subsets

The following functions are used to:

- Extract species-specific datasets
- Remove samples with missing values
- Compute resistance patterns (LPS)
- Filter rare patterns

For each species:

1. Extract spectra and AMR labels from **both datasets**.
2. Concatenate DRIAMS and Marisma samples.
3. Remove samples with missing values.
4. Compute resistance patterns.
5. Filter rare patterns (<10 samples).

This results in a curated dataset for each species.

In [ ]:
# Cell 3: Funciones auxiliares

def extract_species_dataset(payload, species, antibiotics_subset):
    """
    Extrae X y matriz AMR sólo para una especie y subset de antibióticos presentes.
    Devuelve X, y, y la lista de antibióticos realmente disponibles en ese payload.
    """

    X = payload["data"]
    y = payload["label"]
    amr = payload["amr"]
    antibiotics = payload["antibiotics"]

    # Filtrar especie
    mask_species = y == species

    X_species = X[mask_species]
    amr_species = amr[mask_species]

    # Quedarse sólo con antibióticos presentes en este dataset
    available_ab = [ab for ab in antibiotics_subset if ab in antibiotics]

    if len(available_ab) == 0:
        return None, None, []

    ab_idx = [antibiotics.index(ab) for ab in available_ab]

    amr_species = amr_species[:, ab_idx]

    return X_species, amr_species, available_ab


def remove_missing(X, y):
    """
    Elimina cualquier muestra que tenga NaNs en el espectro o en la matriz AMR.
    """

    mask_valid = ~np.isnan(X).any(axis=1) & ~np.isnan(y).any(axis=1)
    return X[mask_valid], y[mask_valid]


def compute_lps_patterns(y):
    """
    Codifica los patrones AMR (0/1) como strings, p.ej. '010', '111',...
    """

    patterns = ["".join(map(str, row.astype(int))) for row in y]
    return np.array(patterns)


def filter_rare_patterns(X, y, min_samples=10):
    """
    Filtra patrones de resistencia que aparecen con menos de min_samples.
    """

    patterns = compute_lps_patterns(y)
    counts = Counter(patterns)
    valid_patterns = {p for p, c in counts.items() if c >= min_samples}
    mask = np.array([p in valid_patterns for p in patterns])

    return X[mask], y[mask], patterns[mask]


def align_matrix(payload, species, antibiotics_final):
    """
    Reconstruye la matriz AMR para una especie, alineada a la lista completa
    de antibióticos (union de DRIAMS + MARISMA), rellenando con NaN cuando falte.
    """

    X = payload["data"]
    y = payload["label"]
    amr = payload["amr"]
    antibiotics = payload["antibiotics"]

    mask = y == species

    X_species = X[mask]
    amr_species = amr[mask]

    Y = np.full((amr_species.shape[0], len(antibiotics_final)), np.nan)

    for j, ab in enumerate(antibiotics_final):
        if ab in antibiotics:
            idx = antibiotics.index(ab)
            Y[:, j] = amr_species[:, idx]

    return X_species, Y

In [ ]:
# Cell 4: Generación de datasets por especie (incluyendo versiones "raw" y origen)

species_datasets = {}

for species, ab_list in species_antibiotics.items():

    print("\nProcessing:", species)

    # Sólo usamos esto para saber qué antibióticos están presentes en cada dataset
    X_d_tmp, y_d_tmp, ab_d = extract_species_dataset(driams_payload, species, ab_list)
    X_m_tmp, y_m_tmp, ab_m = extract_species_dataset(marisma_payload, species, ab_list)

    if X_d_tmp is None and X_m_tmp is None:
        print("No antibiotics available.")
        continue

    # Unión de antibióticos disponibles en DRIAMS y MARISMA
    final_antibiotics = sorted(list(set(ab_d).union(set(ab_m))))
    print("Final antibiotics:", final_antibiotics)

    # Reconstruir matrices alineadas a la lista completa de antibióticos
    X_d, y_d = align_matrix(driams_payload, species, final_antibiotics)
    X_m, y_m = align_matrix(marisma_payload, species, final_antibiotics)

    # Concatenar DRIAMS + MARISMA
    X = np.vstack([X_d, X_m])
    y = np.vstack([y_d, y_m])

    # Guardar copia "raw" (con NaNs y sin filtrar patrones)
    X_raw = X.copy()
    y_raw = y.copy()
    origin_raw = np.array(
        ["DRIAMS"] * X_d.shape[0] +
        ["MARISMA"] * X_m.shape[0]
    )

    # Eliminar muestras con valores missing en espectro o AMR
    X, y = remove_missing(X, y)

    # Filtrar patrones poco frecuentes
    X, y, patterns = filter_rare_patterns(X, y)

    species_datasets[species] = {
        "X": X,                         # limpio (sin NaNs, patrones frecuentes)
        "y": y,                         # limpio
        "patterns": patterns,
        "antibiotics": final_antibiotics,
        "X_raw": X_raw,                 # con NaNs
        "y_raw": y_raw,                 # con NaNs
        "origin_raw": origin_raw        # DRIAMS / MARISMA
    }


Processing: Staphylococcus_Aureus
Final antibiotics: ['Clindamycin', 'Fusidic acid', 'Oxacillin']

Processing: Escherichia_Coli
Final antibiotics: ['Cefepime', 'Ceftriaxone', 'Ciprofloxacin']

Processing: Klebsiella_Pneumoniae
Final antibiotics: ['Ciprofloxacin', 'Imipenem', 'Meropenem']

Processing: Pseudomonas_Aeruginosa
Final antibiotics: ['Ciprofloxacin', 'Imipenem', 'Meropenem']


## Dataset statistics

### Species samples

In [ ]:
rows = []

for species, dataset in species_datasets.items():

    X = dataset["X"]
    y = dataset["y"]

    rows.append({
        "species": species,
        "samples": X.shape[0],
        "spectral_features": X.shape[1],
        "antibiotics": len(dataset["antibiotics"])
    })

species_stats = pd.DataFrame(rows).sort_values("samples", ascending=False)

species_stats

,species,samples,spectral_features,antibiotics
2,Klebsiella_Pneumoniae,18386,6000,3
1,Escherichia_Coli,5616,6000,3
0,Staphylococcus_Aureus,4649,6000,3
3,Pseudomonas_Aeruginosa,3843,6000,3


### Pattern distribution

In [ ]:
pattern_tables = {}

for species, dataset in species_datasets.items():

    patterns = dataset["patterns"]
    pattern_counts = Counter(patterns)

    df = pd.DataFrame({
        "pattern": list(pattern_counts.keys()),
        "count": list(pattern_counts.values())
    }).sort_values("count", ascending=False)

    df["frequency"] = df["count"] / df["count"].sum()

    pattern_tables[species] = df

    print("\n")
    print("="*60)
    print(species)
    print("="*60)

    display(df)



Staphylococcus_Aureus


,pattern,count,frequency
0,000,3260,0.701226
6,001,457,0.098301
2,100,378,0.081308
1,101,211,0.045386
3,010,166,0.035707
4,011,99,0.021295
7,110,48,0.010325
5,111,30,0.006453




Escherichia_Coli


,pattern,count,frequency
1,000,3723,0.662927
2,111,722,0.128561
0,001,701,0.124822
3,110,229,0.040776
6,011,143,0.025463
4,010,76,0.013533
5,101,22,0.003917




Klebsiella_Pneumoniae


,pattern,count,frequency
0,000,13592,0.739258
1,100,4186,0.227673
3,111,359,0.019526
6,110,119,0.006472
4,101,54,0.002937
2,011,38,0.002067
5,010,38,0.002067




Pseudomonas_Aeruginosa


,pattern,count,frequency
0,000,1918,0.499089
2,110,818,0.212855
3,010,440,0.114494
5,111,331,0.086131
1,100,171,0.044496
4,011,165,0.042935


### Antibiotic resistance distributions

In [ ]:
rows = []

for species, dataset in species_datasets.items():

    y = dataset["y"]
    antibiotics = dataset["antibiotics"]

    for i, ab in enumerate(antibiotics):

        resistant = np.sum(y[:, i] == 1)
        susceptible = np.sum(y[:, i] == 0)

        total = resistant + susceptible

        rows.append({
            "species": species,
            "antibiotic": ab,
            "resistant": resistant,
            "susceptible": susceptible,
            "resistance_rate": resistant / total
        })

resistance_stats = pd.DataFrame(rows)

resistance_stats

,species,antibiotic,resistant,susceptible,resistance_rate
0,Staphylococcus_Aureus,Clindamycin,667,3982,0.143472
1,Staphylococcus_Aureus,Fusidic acid,343,4306,0.073779
2,Staphylococcus_Aureus,Oxacillin,797,3852,0.171435
3,Escherichia_Coli,Cefepime,973,4643,0.173255
4,Escherichia_Coli,Ceftriaxone,1170,4446,0.208333
5,Escherichia_Coli,Ciprofloxacin,1588,4028,0.282764
6,Klebsiella_Pneumoniae,Ciprofloxacin,4718,13668,0.256608
7,Klebsiella_Pneumoniae,Imipenem,554,17832,0.030132
8,Klebsiella_Pneumoniae,Meropenem,451,17935,0.024530
9,Pseudomonas_Aeruginosa,Ciprofloxacin,1320,2523,0.343482


### Dropped samples

In [ ]:
for species, d in species_datasets.items():

    X_raw = d["X_raw"]
    y_raw = d["y_raw"]
    origin_raw = d["origin_raw"]

    missing_spectrum = np.isnan(X_raw).any(axis=1)
    missing_amr = np.isnan(y_raw).any(axis=1)

    removed_total = missing_spectrum | missing_amr

    print("\n")
    print("=" * 60)
    print(species)
    print("=" * 60)

    print("Total samples (raw):", len(X_raw))

    print("\nRemoved due to spectrum:")
    print(np.sum(missing_spectrum))

    print("\nRemoved due to AMR:")
    print(np.sum(missing_amr))

    print("\nRemoved total (missing spectrum or AMR):")
    print(np.sum(removed_total))

    removed_origin = origin_raw[removed_total]

    print("\nRemoved from DRIAMS:", np.sum(removed_origin == "DRIAMS"))
    print("Removed from MARISMA:", np.sum(removed_origin == "MARISMA"))



Staphylococcus_Aureus
Total samples (raw): 7224

Removed due to spectrum:
0

Removed due to AMR:
2575

Removed total (missing spectrum or AMR):
2575

Removed from DRIAMS: 326
Removed from MARISMA: 2249


Escherichia_Coli
Total samples (raw): 7716

Removed due to spectrum:
0

Removed due to AMR:
2093

Removed total (missing spectrum or AMR):
2093

Removed from DRIAMS: 507
Removed from MARISMA: 1586


Klebsiella_Pneumoniae
Total samples (raw): 19259

Removed due to spectrum:
0

Removed due to AMR:
867

Removed total (missing spectrum or AMR):
867

Removed from DRIAMS: 423
Removed from MARISMA: 444


Pseudomonas_Aeruginosa
Total samples (raw): 5329

Removed due to spectrum:
0

Removed due to AMR:
1481

Removed total (missing spectrum or AMR):
1481

Removed from DRIAMS: 1384
Removed from MARISMA: 97


### Missing antibiotics

In [ ]:
for species, d in species_datasets.items():

    y = d["y_raw"]
    antibiotics = d["antibiotics"]

    print("\n")
    print("="*60)
    print(species)
    print("="*60)

    for i, ab in enumerate(antibiotics):

        missing = np.sum(np.isnan(y[:, i]))

        print(f"{ab}: missing {missing}")



Staphylococcus_Aureus
Clindamycin: missing 1127
Fusidic acid: missing 1709
Oxacillin: missing 10


Escherichia_Coli
Cefepime: missing 406
Ceftriaxone: missing 1626
Ciprofloxacin: missing 145


Klebsiella_Pneumoniae
Ciprofloxacin: missing 53
Imipenem: missing 219
Meropenem: missing 800


Pseudomonas_Aeruginosa
Ciprofloxacin: missing 162
Imipenem: missing 939
Meropenem: missing 520


### Missing antibiotics / dataset

In [ ]:
for species, d in species_datasets.items():

    y = d["y_raw"]
    antibiotics = d["antibiotics"]
    origin = d["origin_raw"]

    print("\n")
    print("="*60)
    print(species)
    print("="*60)

    for i, ab in enumerate(antibiotics):

        missing_driams = np.sum(np.isnan(y[origin=="DRIAMS", i]))
        missing_marisma = np.sum(np.isnan(y[origin=="MARISMA", i]))

        print(f"{ab}")
        print("   missing DRIAMS:", missing_driams)
        print("   missing MARISMA:", missing_marisma)



Staphylococcus_Aureus
Clindamycin
   missing DRIAMS: 303
   missing MARISMA: 824
Fusidic acid
   missing DRIAMS: 114
   missing MARISMA: 1595
Oxacillin
   missing DRIAMS: 3
   missing MARISMA: 7


Escherichia_Coli
Cefepime
   missing DRIAMS: 405
   missing MARISMA: 1
Ceftriaxone
   missing DRIAMS: 40
   missing MARISMA: 1586
Ciprofloxacin
   missing DRIAMS: 117
   missing MARISMA: 28


Klebsiella_Pneumoniae
Ciprofloxacin
   missing DRIAMS: 31
   missing MARISMA: 22
Imipenem
   missing DRIAMS: 203
   missing MARISMA: 16
Meropenem
   missing DRIAMS: 381
   missing MARISMA: 419


Pseudomonas_Aeruginosa
Ciprofloxacin
   missing DRIAMS: 122
   missing MARISMA: 40
Imipenem
   missing DRIAMS: 901
   missing MARISMA: 38
Meropenem
   missing DRIAMS: 460
   missing MARISMA: 60


## Defining MLP architectures

##

In [ ]:
mlp_lps_params = {
    "Escherichia_Coli": dict(
        hidden_layer_sizes=(481, 24, 500),
        activation="relu",
        solver="adam",
        learning_rate_init=2.4378878006413237e-05
    ),
    "Klebsiella_Pneumoniae": dict(
        hidden_layer_sizes=(94, 313, 99),
        activation="tanh",
        solver="adam",
        learning_rate_init=7.693489492906515e-05
    ),
    "Pseudomonas_Aeruginosa": dict(
        hidden_layer_sizes=(402, 208, 435),
        activation="relu",
        solver="adam",
        learning_rate_init=0.004215968282365892
    ),
    "Staphylococcus_Aureus": dict(
        hidden_layer_sizes=(289, 242, 189),
        activation="identity",
        solver="adam",
        learning_rate_init=0.0008078817205716293
    ),
}

mlp_multi_params = {
    "Escherichia_Coli": dict(
        hidden_layer_sizes=(477, 456, 73),
        activation="tanh",
        solver="adam",
        learning_rate_init=1.7708879893278444e-05
    ),
    "Klebsiella_Pneumoniae": dict(
        hidden_layer_sizes=(75, 90, 318),
        activation="tanh",
        solver="adam",
        learning_rate_init=9.459131060843723e-05
    ),
    "Pseudomonas_Aeruginosa": dict(
        hidden_layer_sizes=(463, 472, 327),
        activation="relu",
        solver="adam",
        learning_rate_init=0.002765076022894333
    ),
    "Staphylococcus_Aureus": dict(
        hidden_layer_sizes=(385, 421, 58),
        activation="identity",
        solver="adam",
        learning_rate_init=2.7646086843997938e-05
    ),
}

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, output_dim, activation):
        super().__init__()

        act_dict = {
            "relu": nn.ReLU(),
            "tanh": nn.Tanh(),
            "logistic": nn.Sigmoid(),
            "identity": nn.Identity()
        }

        layers = []
        prev_dim = input_dim

        for h in hidden_layers:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(act_dict[activation])
            prev_dim = h

        layers.append(nn.Linear(prev_dim, output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

## Training utils

In [ ]:
def evaluate_multilabel_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    wf1s, accs, hls = [], [], []

    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_pred[:, j]

        wf1s.append(f1_score(yt, yp, average="weighted"))
        accs.append(accuracy_score(yt, yp))
        hls.append(hamming_loss(yt, yp))

    return {
        "WF1": float(np.mean(wf1s)),
        "ACC": float(np.mean(accs)),
        "HL": float(np.mean(hls)),
    }


def lps_index_to_multilabel(pred_indices: np.ndarray, class_to_pattern: dict) -> np.ndarray:
    preds = []

    for idx in pred_indices:
        pat = class_to_pattern[int(idx)]
        preds.append([int(c) for c in pat])

    return np.asarray(preds, dtype=int)


def evaluate_lps_patterns(y_true_patterns: np.ndarray,
                          y_pred_indices: np.ndarray,
                          class_to_pattern: dict):
    y_true_multi = np.asarray([[int(c) for c in p] for p in y_true_patterns], dtype=int)
    y_pred_multi = lps_index_to_multilabel(y_pred_indices, class_to_pattern)

    return evaluate_multilabel_metrics(y_true_multi, y_pred_multi)


@torch.no_grad()
def predict_lps_class(model, X):
    model.eval()
    logits = model(X)
    return torch.argmax(logits, dim=1).cpu().numpy()


@torch.no_grad()
def predict_multilabel(model, X):
    model.eval()
    logits = model(X)
    probs = torch.sigmoid(logits)
    return (probs > 0.5).long().cpu().numpy()

In [ ]:
def train_model_with_early_stopping(
    model,
    optimizer,
    criterion,
    X_train,
    y_train,
    device,
    max_epochs=1200,
    patience=50,
    batch_size=128,
    multilabel=False
):
    from torch.utils.data import DataLoader, TensorDataset

    X_np = X_train.cpu().numpy()
    y_np = y_train.cpu().numpy()

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_np,
        y_np,
        test_size=0.1,
        random_state=42
    )

    X_tr = torch.tensor(X_tr, dtype=torch.float32).to(device)
    X_val = torch.tensor(X_val, dtype=torch.float32).to(device)

    if multilabel:
        y_tr = torch.tensor(y_tr, dtype=torch.float32).to(device)
    else:
        y_tr = torch.tensor(y_tr, dtype=torch.long).to(device)

    dataset = TensorDataset(X_tr, y_tr)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    best_state = None
    best_score = -np.inf
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()

        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            logits_val = model(X_val)

        if multilabel:
            probs = torch.sigmoid(logits_val)
            preds = (probs > 0.5).long().cpu().numpy()
            score = evaluate_multilabel_metrics(y_val.astype(int), preds)["WF1"]
        else:
            preds = torch.argmax(logits_val, dim=1).cpu().numpy()
            score = f1_score(y_val, preds, average="weighted")

        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

## Train model

In [ ]:
results = []
per_fold_results = []

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for species, dataset in species_datasets.items():

    print("\n" + "=" * 80)
    print(f"Processing {species}")
    print("=" * 80)

    X_sp = dataset["X"]
    Y_multi = dataset["y"].astype(int)
    LPS_patterns = dataset["patterns"]
    ab_list = dataset["antibiotics"]

    print("Samples:", X_sp.shape[0])
    print("Features:", X_sp.shape[1])
    print("Antibiotics:", ab_list)
    print("Unique LPS patterns:", len(np.unique(LPS_patterns)))

    metrics_lps = []
    metrics_multi = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_sp, LPS_patterns), start=1):

        print(f"\nFold {fold}/{N_FOLDS}")

        X_train = X_sp[train_idx]
        X_test = X_sp[test_idx]

        y_train_multi = Y_multi[train_idx]
        y_test_multi = Y_multi[test_idx]

        pat_train = LPS_patterns[train_idx]
        pat_test = LPS_patterns[test_idx]

        # Mapeo LPS -> clase
        patterns_train_unique = np.unique(pat_train)
        pattern_to_class = {p: i for i, p in enumerate(patterns_train_unique)}
        class_to_pattern = {i: p for p, i in pattern_to_class.items()}

        y_train_lps = np.array([pattern_to_class[p] for p in pat_train], dtype=int)

        # Por seguridad: eliminar del test patrones no vistos en train
        seen_mask_test = np.array([p in pattern_to_class for p in pat_test])

        X_test = X_test[seen_mask_test]
        y_test_multi = y_test_multi[seen_mask_test]
        pat_test = pat_test[seen_mask_test]

        n_classes = len(pattern_to_class)

        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

        y_train_lps_t = torch.tensor(y_train_lps, dtype=torch.long).to(device)
        y_train_multi_t = torch.tensor(y_train_multi, dtype=torch.float32).to(device)

        # ==========================
        # LPS model
        # ==========================
        model_lps = MLP(
            input_dim=X_sp.shape[1],
            hidden_layers=mlp_lps_params[species]["hidden_layer_sizes"],
            output_dim=n_classes,
            activation=mlp_lps_params[species]["activation"]
        ).to(device)

        optimizer_lps = optim.Adam(
            model_lps.parameters(),
            lr=mlp_lps_params[species]["learning_rate_init"]
        )

        criterion_lps = nn.CrossEntropyLoss()

        model_lps = train_model_with_early_stopping(
            model=model_lps,
            optimizer=optimizer_lps,
            criterion=criterion_lps,
            X_train=X_train_t,
            y_train=y_train_lps_t,
            device=device,
            multilabel=False
        )

        pred_class = predict_lps_class(model_lps, X_test_t)
        lps_metrics = evaluate_lps_patterns(pat_test, pred_class, class_to_pattern)
        metrics_lps.append(lps_metrics)

        # ==========================
        # Direct Multilabel model
        # ==========================
        model_multi = MLP(
            input_dim=X_sp.shape[1],
            hidden_layers=mlp_multi_params[species]["hidden_layer_sizes"],
            output_dim=len(ab_list),
            activation=mlp_multi_params[species]["activation"]
        ).to(device)

        optimizer_multi = optim.Adam(
            model_multi.parameters(),
            lr=mlp_multi_params[species]["learning_rate_init"]
        )

        criterion_multi = nn.BCEWithLogitsLoss()

        model_multi = train_model_with_early_stopping(
            model=model_multi,
            optimizer=optimizer_multi,
            criterion=criterion_multi,
            X_train=X_train_t,
            y_train=y_train_multi_t,
            device=device,
            multilabel=True
        )

        pred_multi = predict_multilabel(model_multi, X_test_t)
        multi_metrics = evaluate_multilabel_metrics(
            y_test_multi.astype(int),
            pred_multi.astype(int)
        )
        metrics_multi.append(multi_metrics)

        per_fold_results.append({
            "Species": species,
            "Fold": fold,
            "LPS_WF1": lps_metrics["WF1"],
            "LPS_ACC": lps_metrics["ACC"],
            "LPS_HL": lps_metrics["HL"],
            "Multi_WF1": multi_metrics["WF1"],
            "Multi_ACC": multi_metrics["ACC"],
            "Multi_HL": multi_metrics["HL"],
        })

    df_lps = pd.DataFrame(metrics_lps)
    df_multi = pd.DataFrame(metrics_multi)

    results.append({
        "Species": species,
        "LPS_WF1": f"{df_lps['WF1'].mean():.3f} ± {df_lps['WF1'].std():.3f}",
        "LPS_ACC": f"{df_lps['ACC'].mean():.3f} ± {df_lps['ACC'].std():.3f}",
        "LPS_HL": f"{df_lps['HL'].mean():.3f} ± {df_lps['HL'].std():.3f}",
        "Multi_WF1": f"{df_multi['WF1'].mean():.3f} ± {df_multi['WF1'].std():.3f}",
        "Multi_ACC": f"{df_multi['ACC'].mean():.3f} ± {df_multi['ACC'].std():.3f}",
        "Multi_HL": f"{df_multi['HL'].mean():.3f} ± {df_multi['HL'].std():.3f}",
    })


Processing Staphylococcus_Aureus
Samples: 4649
Features: 6000
Antibiotics: ['Clindamycin', 'Fusidic acid', 'Oxacillin']
Unique LPS patterns: 8

Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5

Processing Escherichia_Coli
Samples: 5616
Features: 6000
Antibiotics: ['Cefepime', 'Ceftriaxone', 'Ciprofloxacin']
Unique LPS patterns: 7

Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5

Processing Klebsiella_Pneumoniae
Samples: 18386
Features: 6000
Antibiotics: ['Ciprofloxacin', 'Imipenem', 'Meropenem']
Unique LPS patterns: 7

Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5

Processing Pseudomonas_Aeruginosa
Samples: 3843
Features: 6000
Antibiotics: ['Ciprofloxacin', 'Imipenem', 'Meropenem']
Unique LPS patterns: 6

Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5


In [ ]:
results

[{'Species': 'Staphylococcus_Aureus',
  'LPS_WF1': '0.882 ± 0.009',
  'LPS_ACC': '0.896 ± 0.013',
  'LPS_HL': '0.104 ± 0.013',
  'Multi_WF1': '0.888 ± 0.005',
  'Multi_ACC': '0.900 ± 0.003',
  'Multi_HL': '0.100 ± 0.003'},
 {'Species': 'Escherichia_Coli',
  'LPS_WF1': '0.832 ± 0.005',
  'LPS_ACC': '0.840 ± 0.005',
  'LPS_HL': '0.160 ± 0.005',
  'Multi_WF1': '0.832 ± 0.007',
  'Multi_ACC': '0.844 ± 0.007',
  'Multi_HL': '0.156 ± 0.007'},
 {'Species': 'Klebsiella_Pneumoniae',
  'LPS_WF1': '0.902 ± 0.003',
  'LPS_ACC': '0.913 ± 0.001',
  'LPS_HL': '0.087 ± 0.001',
  'Multi_WF1': '0.905 ± 0.002',
  'Multi_ACC': '0.912 ± 0.004',
  'Multi_HL': '0.088 ± 0.004'},
 {'Species': 'Pseudomonas_Aeruginosa',
  'LPS_WF1': '0.856 ± 0.012',
  'LPS_ACC': '0.860 ± 0.013',
  'LPS_HL': '0.140 ± 0.013',
  'Multi_WF1': '0.863 ± 0.009',
  'Multi_ACC': '0.871 ± 0.010',
  'Multi_HL': '0.129 ± 0.010'}]

## Imports and configuration

In [15]:
import pickle
import numpy as np
import pandas as pd
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, accuracy_score, hamming_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

driams_path = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/DRIAMS_A_AMR_half_pipeline_23_DRIAMS.pkl"
marisma_path = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/MARISMa_study_MARISMA_half_pipeline.pkl"

with open(driams_path, "rb") as f:
    driams_payload = pickle.load(f)

with open(marisma_path, "rb") as f:
    marisma_payload = pickle.load(f)

print("Datasets loaded.")


def inspect_dataset(payload, name):

    X = payload["data"]
    y_species = payload["label"]
    amr = payload["amr"]
    antibiotics = payload["antibiotics"]

    print(f"\n{name}")
    print("-" * 40)
    print("Spectra shape:", X.shape)
    print("AMR matrix shape:", amr.shape)
    print("Number of antibiotics:", len(antibiotics))
    print("Unique species:", np.unique(y_species))


inspect_dataset(driams_payload, "DRIAMS ABC")
inspect_dataset(marisma_payload, "MARISMA")

Using device: cpu
Datasets loaded.

DRIAMS ABC
----------------------------------------
Spectra shape: (18168, 1600)
AMR matrix shape: (18168, 9)
Number of antibiotics: 9
Unique species: ['Escherichia_Coli' 'Klebsiella_Pneumoniae' 'Pseudomonas_Aeruginosa'
 'Staphylococcus_Aureus']

MARISMA
----------------------------------------
Spectra shape: (21360, 1600)
AMR matrix shape: (21360, 8)
Number of antibiotics: 8
Unique species: ['Escherichia_Coli' 'Klebsiella_Pneumoniae' 'Pseudomonas_Aeruginosa'
 'Staphylococcus_Aureus']


## Species specific antibiotic definition

In [3]:
species_antibiotics = {
    "Staphylococcus_Aureus": [
        "Oxacillin", "Clindamycin", "Fusidic acid"
    ],
    "Escherichia_Coli": [
        "Ciprofloxacin", "Ceftriaxone",
        #"Piperacillin-Tazobactam",
		 "Cefepime"
    ],
    "Klebsiella_Pneumoniae": [
        "Ciprofloxacin", #"Ceftriaxone",
        "Imipenem", "Meropenem"
    ],
    "Pseudomonas_Aeruginosa": [
        "Ciprofloxacin", "Imipenem", "Meropenem"
    ]
}

## Generate species subsets

The following functions are used to:

- Extract species-specific datasets
- Remove samples with missing values
- Compute resistance patterns (LPS)
- Filter rare patterns

For each species:

1. Extract spectra and AMR labels from **both datasets**.
2. Concatenate DRIAMS and Marisma samples.
3. Remove samples with missing values.
4. Compute resistance patterns.
5. Filter rare patterns (<10 samples).

This results in a curated dataset for each species.

In [4]:
# Cell 3: Funciones auxiliares

def extract_species_dataset(payload, species, antibiotics_subset):
    """
    Extrae X y matriz AMR sólo para una especie y subset de antibióticos presentes.
    Devuelve X, y, y la lista de antibióticos realmente disponibles en ese payload.
    """

    X = payload["data"]
    y = payload["label"]
    amr = payload["amr"]
    antibiotics = payload["antibiotics"]

    # Filtrar especie
    mask_species = y == species

    X_species = X[mask_species]
    amr_species = amr[mask_species]

    # Quedarse sólo con antibióticos presentes en este dataset
    available_ab = [ab for ab in antibiotics_subset if ab in antibiotics]

    if len(available_ab) == 0:
        return None, None, []

    ab_idx = [antibiotics.index(ab) for ab in available_ab]

    amr_species = amr_species[:, ab_idx]

    return X_species, amr_species, available_ab


def remove_missing(X, y):
    """
    Elimina cualquier muestra que tenga NaNs en el espectro o en la matriz AMR.
    """

    mask_valid = ~np.isnan(X).any(axis=1) & ~np.isnan(y).any(axis=1)
    return X[mask_valid], y[mask_valid]


def compute_lps_patterns(y):
    """
    Codifica los patrones AMR (0/1) como strings, p.ej. '010', '111',...
    """

    patterns = ["".join(map(str, row.astype(int))) for row in y]
    return np.array(patterns)


def filter_rare_patterns(X, y, min_samples=10):
    """
    Filtra patrones de resistencia que aparecen con menos de min_samples.
    """

    patterns = compute_lps_patterns(y)
    counts = Counter(patterns)
    valid_patterns = {p for p, c in counts.items() if c >= min_samples}
    mask = np.array([p in valid_patterns for p in patterns])

    return X[mask], y[mask], patterns[mask]


def align_matrix(payload, species, antibiotics_final):
    """
    Reconstruye la matriz AMR para una especie, alineada a la lista completa
    de antibióticos (union de DRIAMS + MARISMA), rellenando con NaN cuando falte.
    """

    X = payload["data"]
    y = payload["label"]
    amr = payload["amr"]
    antibiotics = payload["antibiotics"]

    mask = y == species

    X_species = X[mask]
    amr_species = amr[mask]

    Y = np.full((amr_species.shape[0], len(antibiotics_final)), np.nan)

    for j, ab in enumerate(antibiotics_final):
        if ab in antibiotics:
            idx = antibiotics.index(ab)
            Y[:, j] = amr_species[:, idx]

    return X_species, Y

In [5]:
# Cell 4: Generación de datasets por especie (incluyendo versiones "raw" y origen)

species_datasets = {}

for species, ab_list in species_antibiotics.items():

    print("\nProcessing:", species)

    # Sólo usamos esto para saber qué antibióticos están presentes en cada dataset
    X_d_tmp, y_d_tmp, ab_d = extract_species_dataset(driams_payload, species, ab_list)
    X_m_tmp, y_m_tmp, ab_m = extract_species_dataset(marisma_payload, species, ab_list)

    if X_d_tmp is None and X_m_tmp is None:
        print("No antibiotics available.")
        continue

    # Unión de antibióticos disponibles en DRIAMS y MARISMA
    final_antibiotics = sorted(list(set(ab_d).union(set(ab_m))))
    print("Final antibiotics:", final_antibiotics)

    # Reconstruir matrices alineadas a la lista completa de antibióticos
    X_d, y_d = align_matrix(driams_payload, species, final_antibiotics)
    X_m, y_m = align_matrix(marisma_payload, species, final_antibiotics)

    # Concatenar DRIAMS + MARISMA
    X = np.vstack([X_d, X_m])
    y = np.vstack([y_d, y_m])

    # Guardar copia "raw" (con NaNs y sin filtrar patrones)
    X_raw = X.copy()
    y_raw = y.copy()
    origin_raw = np.array(
        ["DRIAMS"] * X_d.shape[0] +
        ["MARISMA"] * X_m.shape[0]
    )

    # Eliminar muestras con valores missing en espectro o AMR
    X, y = remove_missing(X, y)

    # Filtrar patrones poco frecuentes
    X, y, patterns = filter_rare_patterns(X, y)

    species_datasets[species] = {
        "X": X,                         # limpio (sin NaNs, patrones frecuentes)
        "y": y,                         # limpio
        "patterns": patterns,
        "antibiotics": final_antibiotics,
        "X_raw": X_raw,                 # con NaNs
        "y_raw": y_raw,                 # con NaNs
        "origin_raw": origin_raw        # DRIAMS / MARISMA
    }


Processing: Staphylococcus_Aureus
Final antibiotics: ['Clindamycin', 'Fusidic acid', 'Oxacillin']

Processing: Escherichia_Coli
Final antibiotics: ['Cefepime', 'Ceftriaxone', 'Ciprofloxacin']

Processing: Klebsiella_Pneumoniae
Final antibiotics: ['Ciprofloxacin', 'Imipenem', 'Meropenem']

Processing: Pseudomonas_Aeruginosa
Final antibiotics: ['Ciprofloxacin', 'Imipenem', 'Meropenem']


## Dataset statistics

### Species samples

In [6]:
rows = []

for species, dataset in species_datasets.items():

    X = dataset["X"]
    y = dataset["y"]

    rows.append({
        "species": species,
        "samples": X.shape[0],
        "spectral_features": X.shape[1],
        "antibiotics": len(dataset["antibiotics"])
    })

species_stats = pd.DataFrame(rows).sort_values("samples", ascending=False)

species_stats

,species,samples,spectral_features,antibiotics
2,Klebsiella_Pneumoniae,18386,1600,3
1,Escherichia_Coli,5616,1600,3
0,Staphylococcus_Aureus,4649,1600,3
3,Pseudomonas_Aeruginosa,3843,1600,3


### Pattern distribution

In [7]:
pattern_tables = {}

for species, dataset in species_datasets.items():

    patterns = dataset["patterns"]
    pattern_counts = Counter(patterns)

    df = pd.DataFrame({
        "pattern": list(pattern_counts.keys()),
        "count": list(pattern_counts.values())
    }).sort_values("count", ascending=False)

    df["frequency"] = df["count"] / df["count"].sum()

    pattern_tables[species] = df

    print("\n")
    print("="*60)
    print(species)
    print("="*60)

    display(df)



Staphylococcus_Aureus


,pattern,count,frequency
0,000,3260,0.701226
6,001,457,0.098301
2,100,378,0.081308
1,101,211,0.045386
3,010,166,0.035707
4,011,99,0.021295
7,110,48,0.010325
5,111,30,0.006453




Escherichia_Coli


,pattern,count,frequency
1,000,3723,0.662927
2,111,722,0.128561
0,001,701,0.124822
3,110,229,0.040776
6,011,143,0.025463
4,010,76,0.013533
5,101,22,0.003917




Klebsiella_Pneumoniae


,pattern,count,frequency
0,000,13592,0.739258
1,100,4186,0.227673
3,111,359,0.019526
6,110,119,0.006472
4,101,54,0.002937
2,011,38,0.002067
5,010,38,0.002067




Pseudomonas_Aeruginosa


,pattern,count,frequency
0,000,1918,0.499089
2,110,818,0.212855
3,010,440,0.114494
5,111,331,0.086131
1,100,171,0.044496
4,011,165,0.042935


### Antibiotic resistance distributions

In [8]:
rows = []

for species, dataset in species_datasets.items():

    y = dataset["y"]
    antibiotics = dataset["antibiotics"]

    for i, ab in enumerate(antibiotics):

        resistant = np.sum(y[:, i] == 1)
        susceptible = np.sum(y[:, i] == 0)

        total = resistant + susceptible

        rows.append({
            "species": species,
            "antibiotic": ab,
            "resistant": resistant,
            "susceptible": susceptible,
            "resistance_rate": resistant / total
        })

resistance_stats = pd.DataFrame(rows)

resistance_stats

,species,antibiotic,resistant,susceptible,resistance_rate
0,Staphylococcus_Aureus,Clindamycin,667,3982,0.143472
1,Staphylococcus_Aureus,Fusidic acid,343,4306,0.073779
2,Staphylococcus_Aureus,Oxacillin,797,3852,0.171435
3,Escherichia_Coli,Cefepime,973,4643,0.173255
4,Escherichia_Coli,Ceftriaxone,1170,4446,0.208333
5,Escherichia_Coli,Ciprofloxacin,1588,4028,0.282764
6,Klebsiella_Pneumoniae,Ciprofloxacin,4718,13668,0.256608
7,Klebsiella_Pneumoniae,Imipenem,554,17832,0.030132
8,Klebsiella_Pneumoniae,Meropenem,451,17935,0.024530
9,Pseudomonas_Aeruginosa,Ciprofloxacin,1320,2523,0.343482


### Dropped samples

In [9]:
for species, d in species_datasets.items():

    X_raw = d["X_raw"]
    y_raw = d["y_raw"]
    origin_raw = d["origin_raw"]

    missing_spectrum = np.isnan(X_raw).any(axis=1)
    missing_amr = np.isnan(y_raw).any(axis=1)

    removed_total = missing_spectrum | missing_amr

    print("\n")
    print("=" * 60)
    print(species)
    print("=" * 60)

    print("Total samples (raw):", len(X_raw))

    print("\nRemoved due to spectrum:")
    print(np.sum(missing_spectrum))

    print("\nRemoved due to AMR:")
    print(np.sum(missing_amr))

    print("\nRemoved total (missing spectrum or AMR):")
    print(np.sum(removed_total))

    removed_origin = origin_raw[removed_total]

    print("\nRemoved from DRIAMS:", np.sum(removed_origin == "DRIAMS"))
    print("Removed from MARISMA:", np.sum(removed_origin == "MARISMA"))



Staphylococcus_Aureus
Total samples (raw): 7224

Removed due to spectrum:
0

Removed due to AMR:
2575

Removed total (missing spectrum or AMR):
2575

Removed from DRIAMS: 326
Removed from MARISMA: 2249


Escherichia_Coli
Total samples (raw): 7716

Removed due to spectrum:
0

Removed due to AMR:
2093

Removed total (missing spectrum or AMR):
2093

Removed from DRIAMS: 507
Removed from MARISMA: 1586


Klebsiella_Pneumoniae
Total samples (raw): 19259

Removed due to spectrum:
0

Removed due to AMR:
867

Removed total (missing spectrum or AMR):
867

Removed from DRIAMS: 423
Removed from MARISMA: 444


Pseudomonas_Aeruginosa
Total samples (raw): 5329

Removed due to spectrum:
0

Removed due to AMR:
1481

Removed total (missing spectrum or AMR):
1481

Removed from DRIAMS: 1384
Removed from MARISMA: 97


### Missing antibiotics

In [10]:
for species, d in species_datasets.items():

    y = d["y_raw"]
    antibiotics = d["antibiotics"]

    print("\n")
    print("="*60)
    print(species)
    print("="*60)

    for i, ab in enumerate(antibiotics):

        missing = np.sum(np.isnan(y[:, i]))

        print(f"{ab}: missing {missing}")



Staphylococcus_Aureus
Clindamycin: missing 1127
Fusidic acid: missing 1709
Oxacillin: missing 10


Escherichia_Coli
Cefepime: missing 406
Ceftriaxone: missing 1626
Ciprofloxacin: missing 145


Klebsiella_Pneumoniae
Ciprofloxacin: missing 53
Imipenem: missing 219
Meropenem: missing 800


Pseudomonas_Aeruginosa
Ciprofloxacin: missing 162
Imipenem: missing 939
Meropenem: missing 520


### Missing antibiotics / dataset

In [11]:
for species, d in species_datasets.items():

    y = d["y_raw"]
    antibiotics = d["antibiotics"]
    origin = d["origin_raw"]

    print("\n")
    print("="*60)
    print(species)
    print("="*60)

    for i, ab in enumerate(antibiotics):

        missing_driams = np.sum(np.isnan(y[origin=="DRIAMS", i]))
        missing_marisma = np.sum(np.isnan(y[origin=="MARISMA", i]))

        print(f"{ab}")
        print("   missing DRIAMS:", missing_driams)
        print("   missing MARISMA:", missing_marisma)



Staphylococcus_Aureus
Clindamycin
   missing DRIAMS: 303
   missing MARISMA: 824
Fusidic acid
   missing DRIAMS: 114
   missing MARISMA: 1595
Oxacillin
   missing DRIAMS: 3
   missing MARISMA: 7


Escherichia_Coli
Cefepime
   missing DRIAMS: 405
   missing MARISMA: 1
Ceftriaxone
   missing DRIAMS: 40
   missing MARISMA: 1586
Ciprofloxacin
   missing DRIAMS: 117
   missing MARISMA: 28


Klebsiella_Pneumoniae
Ciprofloxacin
   missing DRIAMS: 31
   missing MARISMA: 22
Imipenem
   missing DRIAMS: 203
   missing MARISMA: 16
Meropenem
   missing DRIAMS: 381
   missing MARISMA: 419


Pseudomonas_Aeruginosa
Ciprofloxacin
   missing DRIAMS: 122
   missing MARISMA: 40
Imipenem
   missing DRIAMS: 901
   missing MARISMA: 38
Meropenem
   missing DRIAMS: 460
   missing MARISMA: 60


## Defining MLP architectures

##

In [12]:
mlp_lps_params = {
    "Escherichia_Coli": dict(
        hidden_layer_sizes=(481, 24, 500),
        activation="relu",
        solver="adam",
        learning_rate_init=2.4378878006413237e-05
    ),
    "Klebsiella_Pneumoniae": dict(
        hidden_layer_sizes=(94, 313, 99),
        activation="tanh",
        solver="adam",
        learning_rate_init=7.693489492906515e-05
    ),
    "Pseudomonas_Aeruginosa": dict(
        hidden_layer_sizes=(402, 208, 435),
        activation="relu",
        solver="adam",
        learning_rate_init=0.004215968282365892
    ),
    "Staphylococcus_Aureus": dict(
        hidden_layer_sizes=(289, 242, 189),
        activation="identity",
        solver="adam",
        learning_rate_init=0.0008078817205716293
    ),
}

mlp_multi_params = {
    "Escherichia_Coli": dict(
        hidden_layer_sizes=(477, 456, 73),
        activation="tanh",
        solver="adam",
        learning_rate_init=1.7708879893278444e-05
    ),
    "Klebsiella_Pneumoniae": dict(
        hidden_layer_sizes=(75, 90, 318),
        activation="tanh",
        solver="adam",
        learning_rate_init=9.459131060843723e-05
    ),
    "Pseudomonas_Aeruginosa": dict(
        hidden_layer_sizes=(463, 472, 327),
        activation="relu",
        solver="adam",
        learning_rate_init=0.002765076022894333
    ),
    "Staphylococcus_Aureus": dict(
        hidden_layer_sizes=(385, 421, 58),
        activation="identity",
        solver="adam",
        learning_rate_init=2.7646086843997938e-05
    ),
}

In [13]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, output_dim, activation):
        super().__init__()

        act_dict = {
            "relu": nn.ReLU(),
            "tanh": nn.Tanh(),
            "logistic": nn.Sigmoid(),
            "identity": nn.Identity()
        }

        layers = []
        prev_dim = input_dim

        for h in hidden_layers:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(act_dict[activation])
            prev_dim = h

        layers.append(nn.Linear(prev_dim, output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

## Training utils

In [14]:
def evaluate_multilabel_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    wf1s, accs, hls = [], [], []

    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_pred[:, j]

        wf1s.append(f1_score(yt, yp, average="weighted"))
        accs.append(accuracy_score(yt, yp))
        hls.append(hamming_loss(yt, yp))

    return {
        "WF1": float(np.mean(wf1s)),
        "ACC": float(np.mean(accs)),
        "HL": float(np.mean(hls)),
    }


def lps_index_to_multilabel(pred_indices: np.ndarray, class_to_pattern: dict) -> np.ndarray:
    preds = []

    for idx in pred_indices:
        pat = class_to_pattern[int(idx)]
        preds.append([int(c) for c in pat])

    return np.asarray(preds, dtype=int)


def evaluate_lps_patterns(y_true_patterns: np.ndarray,
                          y_pred_indices: np.ndarray,
                          class_to_pattern: dict):
    y_true_multi = np.asarray([[int(c) for c in p] for p in y_true_patterns], dtype=int)
    y_pred_multi = lps_index_to_multilabel(y_pred_indices, class_to_pattern)

    return evaluate_multilabel_metrics(y_true_multi, y_pred_multi)


@torch.no_grad()
def predict_lps_class(model, X):
    model.eval()
    logits = model(X)
    return torch.argmax(logits, dim=1).cpu().numpy()


@torch.no_grad()
def predict_multilabel(model, X):
    model.eval()
    logits = model(X)
    probs = torch.sigmoid(logits)
    return (probs > 0.5).long().cpu().numpy()

In [14]:
def train_model_with_early_stopping(
    model,
    optimizer,
    criterion,
    X_train,
    y_train,
    device,
    max_epochs=1200,
    patience=50,
    batch_size=128,
    multilabel=False
):
    from torch.utils.data import DataLoader, TensorDataset

    X_np = X_train.cpu().numpy()
    y_np = y_train.cpu().numpy()

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_np,
        y_np,
        test_size=0.1,
        random_state=42
    )

    X_tr = torch.tensor(X_tr, dtype=torch.float32).to(device)
    X_val = torch.tensor(X_val, dtype=torch.float32).to(device)

    if multilabel:
        y_tr = torch.tensor(y_tr, dtype=torch.float32).to(device)
    else:
        y_tr = torch.tensor(y_tr, dtype=torch.long).to(device)

    dataset = TensorDataset(X_tr, y_tr)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    best_state = None
    best_score = -np.inf
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()

        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            logits_val = model(X_val)

        if multilabel:
            probs = torch.sigmoid(logits_val)
            preds = (probs > 0.5).long().cpu().numpy()
            score = evaluate_multilabel_metrics(y_val.astype(int), preds)["WF1"]
        else:
            preds = torch.argmax(logits_val, dim=1).cpu().numpy()
            score = f1_score(y_val, preds, average="weighted")

        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

## Train model

In [16]:
results = []
per_fold_results = []

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for species, dataset in species_datasets.items():

    print("\n" + "=" * 80)
    print(f"Processing {species}")
    print("=" * 80)

    X_sp = dataset["X"]
    Y_multi = dataset["y"].astype(int)
    LPS_patterns = dataset["patterns"]
    ab_list = dataset["antibiotics"]

    print("Samples:", X_sp.shape[0])
    print("Features:", X_sp.shape[1])
    print("Antibiotics:", ab_list)
    print("Unique LPS patterns:", len(np.unique(LPS_patterns)))

    metrics_lps = []
    metrics_multi = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_sp, LPS_patterns), start=1):

        print(f"\nFold {fold}/{N_FOLDS}")

        X_train = X_sp[train_idx]
        X_test = X_sp[test_idx]

        y_train_multi = Y_multi[train_idx]
        y_test_multi = Y_multi[test_idx]

        pat_train = LPS_patterns[train_idx]
        pat_test = LPS_patterns[test_idx]

        # ==========================
        # StandardScaler
        # ==========================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # ==========================
        # LPS mapping
        # ==========================
        patterns_train_unique = np.unique(pat_train)
        pattern_to_class = {p: i for i, p in enumerate(patterns_train_unique)}
        class_to_pattern = {i: p for p, i in pattern_to_class.items()}

        y_train_lps = np.array([pattern_to_class[p] for p in pat_train], dtype=int)

        # Eliminar patrones no vistos
        seen_mask_test = np.array([p in pattern_to_class for p in pat_test])

        X_test = X_test[seen_mask_test]
        y_test_multi = y_test_multi[seen_mask_test]
        pat_test = pat_test[seen_mask_test]

        n_classes = len(pattern_to_class)

        # ==========================
        # Convert to torch
        # ==========================
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

        y_train_lps_t = torch.tensor(y_train_lps, dtype=torch.long).to(device)
        y_train_multi_t = torch.tensor(y_train_multi, dtype=torch.float32).to(device)

        # ==========================
        # LPS model
        # ==========================
        model_lps = MLP(
            input_dim=X_sp.shape[1],
            hidden_layers=mlp_lps_params[species]["hidden_layer_sizes"],
            output_dim=n_classes,
            activation=mlp_lps_params[species]["activation"]
        ).to(device)

        optimizer_lps = optim.Adam(
            model_lps.parameters(),
            lr=mlp_lps_params[species]["learning_rate_init"]
        )

        criterion_lps = nn.CrossEntropyLoss()

        model_lps = train_model_with_early_stopping(
            model=model_lps,
            optimizer=optimizer_lps,
            criterion=criterion_lps,
            X_train=X_train_t,
            y_train=y_train_lps_t,
            device=device,
            multilabel=False
        )

        pred_class = predict_lps_class(model_lps, X_test_t)
        lps_metrics = evaluate_lps_patterns(pat_test, pred_class, class_to_pattern)
        metrics_lps.append(lps_metrics)

        # ==========================
        # Direct Multilabel model
        # ==========================
        model_multi = MLP(
            input_dim=X_sp.shape[1],
            hidden_layers=mlp_multi_params[species]["hidden_layer_sizes"],
            output_dim=len(ab_list),
            activation=mlp_multi_params[species]["activation"]
        ).to(device)

        optimizer_multi = optim.Adam(
            model_multi.parameters(),
            lr=mlp_multi_params[species]["learning_rate_init"]
        )

        criterion_multi = nn.BCEWithLogitsLoss()

        model_multi = train_model_with_early_stopping(
            model=model_multi,
            optimizer=optimizer_multi,
            criterion=criterion_multi,
            X_train=X_train_t,
            y_train=y_train_multi_t,
            device=device,
            multilabel=True
        )

        pred_multi = predict_multilabel(model_multi, X_test_t)
        multi_metrics = evaluate_multilabel_metrics(
            y_test_multi.astype(int),
            pred_multi.astype(int)
        )

        metrics_multi.append(multi_metrics)

        per_fold_results.append({
            "Species": species,
            "Fold": fold,
            "LPS_WF1": lps_metrics["WF1"],
            "LPS_ACC": lps_metrics["ACC"],
            "LPS_HL": lps_metrics["HL"],
            "Multi_WF1": multi_metrics["WF1"],
            "Multi_ACC": multi_metrics["ACC"],
            "Multi_HL": multi_metrics["HL"],
        })

    df_lps = pd.DataFrame(metrics_lps)
    df_multi = pd.DataFrame(metrics_multi)

    results.append({
        "Species": species,
        "LPS_WF1": f"{df_lps['WF1'].mean():.3f} ± {df_lps['WF1'].std():.3f}",
        "LPS_ACC": f"{df_lps['ACC'].mean():.3f} ± {df_lps['ACC'].std():.3f}",
        "LPS_HL": f"{df_lps['HL'].mean():.3f} ± {df_lps['HL'].std():.3f}",
        "Multi_WF1": f"{df_multi['WF1'].mean():.3f} ± {df_multi['WF1'].std():.3f}",
        "Multi_ACC": f"{df_multi['ACC'].mean():.3f} ± {df_multi['ACC'].std():.3f}",
        "Multi_HL": f"{df_multi['HL'].mean():.3f} ± {df_multi['HL'].std():.3f}",
    })


Processing Staphylococcus_Aureus
Samples: 4649
Features: 1600
Antibiotics: ['Clindamycin', 'Fusidic acid', 'Oxacillin']
Unique LPS patterns: 8

Fold 1/5


NameError: name 'train_model_with_early_stopping' is not defined

In [ ]:
results

[{'Species': 'Staphylococcus_Aureus',
  'LPS_WF1': '0.882 ± 0.009',
  'LPS_ACC': '0.896 ± 0.013',
  'LPS_HL': '0.104 ± 0.013',
  'Multi_WF1': '0.888 ± 0.005',
  'Multi_ACC': '0.900 ± 0.003',
  'Multi_HL': '0.100 ± 0.003'},
 {'Species': 'Escherichia_Coli',
  'LPS_WF1': '0.832 ± 0.005',
  'LPS_ACC': '0.840 ± 0.005',
  'LPS_HL': '0.160 ± 0.005',
  'Multi_WF1': '0.832 ± 0.007',
  'Multi_ACC': '0.844 ± 0.007',
  'Multi_HL': '0.156 ± 0.007'},
 {'Species': 'Klebsiella_Pneumoniae',
  'LPS_WF1': '0.902 ± 0.003',
  'LPS_ACC': '0.913 ± 0.001',
  'LPS_HL': '0.087 ± 0.001',
  'Multi_WF1': '0.905 ± 0.002',
  'Multi_ACC': '0.912 ± 0.004',
  'Multi_HL': '0.088 ± 0.004'},
 {'Species': 'Pseudomonas_Aeruginosa',
  'LPS_WF1': '0.856 ± 0.012',
  'LPS_ACC': '0.860 ± 0.013',
  'LPS_HL': '0.140 ± 0.013',
  'Multi_WF1': '0.863 ± 0.009',
  'Multi_ACC': '0.871 ± 0.010',
  'Multi_HL': '0.129 ± 0.010'}]